# Authenta Python SDK — Complete Examples

This notebook walks through every service offered by the Authenta SDK:

| Section | Service |
| :-- | :-- |
| 1 | Setup & Initialization |
| 2 | AC-1 — AI-Generated Image Detection |
| 3 | DF-1 — Deepfake Video Detection |
| 4 | FI-1 — Liveness Detection |
| 5 | FI-1 — Face Swap Detection |
| 6 | FI-1 — Face Similarity Check |
| 7 | FI-1 — Manual Polling (`auto_polling=False`) |
| 8 | Media Management (get, list, delete) |
| 9 | Visualization (heatmaps, bounding boxes) |
| 10 | Async Client Examples |

---
## 1. Setup & Initialization

In [ ]:
import sys
import os
import json
import requests

# Add the src directory so we can import the SDK locally
sys.path.insert(0, os.path.abspath('../src'))

from authenta.authenta_client import AuthentaClient
from authenta import (
    AuthentaError,
    AuthenticationError,
    QuotaExceededError,
    InsufficientCreditsError,
)

# Replace with your credentials
CLIENT_ID     = "YOUR_CLIENT_ID"
CLIENT_SECRET = "YOUR_CLIENT_SECRET"
BASE_URL      = "https://platform.authenta.ai"

client = AuthentaClient(
    base_url=BASE_URL,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
)

print("Client initialized successfully.")

---
## 2. AC-1 — AI-Generated Image Detection

Detect whether an image was created by generative AI (Midjourney, Stable Diffusion, DALL·E, etc.) or manipulated with editing tools.

### 2a. One-call detection (upload + wait)

In [ ]:
IMAGE_PATH = "../data_samples/nano_img2.png"

media = client.process(IMAGE_PATH, model_type="AC-1")

print(f"Media ID  : {media['mid']}")
print(f"Status    : {media['status']}")
print(f"Is Fake   : {media.get('fake')}")
print(f"Result URL: {media.get('resultURL')}")
print(f"Heatmap   : {media.get('heatmapURL')}")

### 2b. Two-step: upload now, poll later

In [ ]:
# Step 1 — upload
upload_meta = client.upload_file(IMAGE_PATH, model_type="AC-1")
mid = upload_meta["mid"]
print(f"Uploaded. Media ID: {mid}  Status: {upload_meta['status']}")

# Step 2 — poll until done
media = client.wait_for_media(mid, interval=5.0, timeout=300.0)
print(f"Final Status: {media['status']}")
print(f"Is Fake     : {media.get('fake')}")

### 2c. Error handling

In [ ]:
try:
    media = client.process(IMAGE_PATH, model_type="AC-1")
    print(f"Is Fake: {media.get('fake')}")
except AuthenticationError:
    print("Authentication failed — check your credentials.")
except QuotaExceededError:
    print("Quota exceeded — upgrade your plan.")
except InsufficientCreditsError:
    print("Not enough credits.")
except TimeoutError as e:
    print(f"Timed out: {e}")
except AuthentaError as e:
    print(f"API error [{e.code}]: {e.message}")

---
## 3. DF-1 — Deepfake Video Detection

Detect face swaps, reenactments, and other facial manipulations in video content.

### 3a. One-call detection

In [ ]:
VIDEO_PATH = "../data_samples/test_00000121.mp4"

media = client.process(VIDEO_PATH, model_type="DF-1")

print(f"Media ID     : {media['mid']}")
print(f"Status       : {media['status']}")
print(f"Is Fake      : {media.get('fake')}")
print(f"Participants : {len(media.get('participants', []))} face(s) detected")

### 3b. Inspect per-participant results

In [ ]:
for idx, participant in enumerate(media.get("participants", [])):
    print(f"\nParticipant {idx}")
    print(f"  Fake        : {participant.get('fake')}")
    print(f"  Confidence  : {participant.get('confidence')}")
    print(f"  Heatmap URL : {participant.get('heatmapURL')}")

### 3c. Two-step with custom timeout

In [ ]:
# Step 1 — upload (longer videos may take time to process)
upload_meta = client.upload_file(VIDEO_PATH, model_type="DF-1")
mid = upload_meta["mid"]
print(f"Uploaded. Media ID: {mid}")

# Step 2 — poll with a 15-minute timeout and 10-second interval
media = client.wait_for_media(mid, interval=10.0, timeout=900.0)
print(f"Status  : {media['status']}")
print(f"Is Fake : {media.get('fake')}")

---
## 4. FI-1 — Liveness Detection

Verify whether a face in an image or video is real or a presentation attack (printed photo, screen replay, deepfake).

### 4a. Liveness on a video

In [ ]:
LIVENESS_VIDEO = "../data_samples/face_live_video/real/sample_real.mp4"

media = client.face_intelligence(
    path=LIVENESS_VIDEO,
    model_type="FI-1",
    livenessCheck=True,
)

print(f"Media ID : {media['mid']}")
print(f"Status   : {media['status']}")
print(f"Liveness : {media.get('isLiveness')}")

### 4b. Batch liveness on a folder of videos

In [ ]:
video_dir = "../data_samples/face_live_video/"
results = []

for label in ["real", "fake"]:
    folder = os.path.join(video_dir, label)
    if not os.path.isdir(folder):
        continue
    for filename in os.listdir(folder):
        if not filename.lower().endswith((".mp4", ".mov")):
            continue
        video_path = os.path.join(folder, filename)
        try:
            print(f"Processing [{label}]: {filename}")
            media = client.face_intelligence(
                path=video_path,
                model_type="FI-1",
                livenessCheck=True,
            )
            results.append({
                "file"      : filename,
                "label"     : label,
                "status"    : media["status"],
                "isLiveness": media.get("isLiveness"),
            })
        except Exception as e:
            print(f"  Error: {e}")

print("\n--- Liveness Results ---")
for r in results:
    print(f"  [{r['label']:4s}] {r['file']:40s}  isLiveness={r['isLiveness']}")

### 4c. Liveness on an image

In [ ]:
LIVENESS_IMAGE = "../data_samples/face_live_images/real/sample_real.jpg"

media = client.face_intelligence(
    path=LIVENESS_IMAGE,
    model_type="FI-1",
    livenessCheck=True,
)

print(f"Status   : {media['status']}")
print(f"Liveness : {media.get('isLiveness')}")

---
## 5. FI-1 — Face Swap Detection (Video Only)

Detect whether a face in a video has been swapped or replaced using AI.

### 5a. Single video

In [ ]:
FACESWAP_VIDEO = "../data_samples/faceswap/fake/sample_swap.mp4"

media = client.face_intelligence(
    path=FACESWAP_VIDEO,
    model_type="FI-1",
    faceswapCheck=True,
)

print(f"Media ID  : {media['mid']}")
print(f"Status    : {media['status']}")
print(f"Face Swap : {media.get('isDeepFake')}")

### 5b. Batch face swap detection

In [ ]:
swap_dir = "../data_samples/faceswap/"
swap_results = []

for label in ["real", "fake"]:
    folder = os.path.join(swap_dir, label)
    if not os.path.isdir(folder):
        continue
    for filename in os.listdir(folder):
        if not filename.lower().endswith((".mp4", ".mov")):
            continue
        video_path = os.path.join(folder, filename)
        try:
            print(f"Processing [{label}]: {filename}")
            media = client.face_intelligence(
                path=video_path,
                model_type="FI-1",
                faceswapCheck=True,
            )
            swap_results.append({
                "file"      : filename,
                "label"     : label,
                "isDeepFake": media.get("isDeepFake"),
            })
        except Exception as e:
            print(f"  Error: {e}")

print("\n--- Face Swap Results ---")
for r in swap_results:
    print(f"  [{r['label']:4s}] {r['file']:40s}  isDeepFake={r['isDeepFake']}")

### 5c. Error: faceswapCheck on an image (raises ValueError)

In [ ]:
try:
    media = client.face_intelligence(
        path="../data_samples/face_similiar/person_1/A.jpeg",
        model_type="FI-1",
        faceswapCheck=True,   # invalid for images
    )
except ValueError as e:
    print(f"Caught expected error: {e}")

---
## 6. FI-1 — Face Similarity Check (Image Only)

Compare two face images and determine whether they belong to the same person.

### 6a. Single pair

In [ ]:
IMAGE_A = "../data_samples/face_similiar/person_1/A.jpeg"
IMAGE_B = "../data_samples/face_similiar/person_1/B.jpeg"

media = client.face_intelligence(
    path=IMAGE_A,
    reference_img_path=IMAGE_B,
    model_type="FI-1",
    faceSimilarityCheck=True,
)

print(f"Media ID         : {media['mid']}")
print(f"Status           : {media['status']}")
print(f"Same Person      : {media.get('isSimilar')}")
print(f"Similarity Score : {media.get('similarityScore')}")

### 6b. Batch similarity across persons

In [ ]:
similarity_dir = "../data_samples/face_similiar/"
similarity_results = []

for person_folder in sorted(os.listdir(similarity_dir)):
    folder_path = os.path.join(similarity_dir, person_folder)
    if not os.path.isdir(folder_path) or person_folder.startswith('.'):
        continue

    files = sorted(
        f for f in os.listdir(folder_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    )
    if len(files) < 2:
        continue

    img_a = os.path.join(folder_path, files[0])
    img_b = os.path.join(folder_path, files[1])

    try:
        print(f"Comparing {person_folder}: {files[0]} vs {files[1]}")
        media = client.face_intelligence(
            path=img_a,
            reference_img_path=img_b,
            model_type="FI-1",
            faceSimilarityCheck=True,
        )
        similarity_results.append({
            "person"    : person_folder,
            "isSimilar" : media.get("isSimilar"),
            "score"     : media.get("similarityScore"),
        })
    except Exception as e:
        print(f"  Error: {e}")

print("\n--- Similarity Results ---")
for r in similarity_results:
    print(f"  {r['person']:12s}  isSimilar={r['isSimilar']}  score={r['score']}")

### 6c. Error: missing reference_img_path

In [ ]:
try:
    media = client.face_intelligence(
        path=IMAGE_A,
        model_type="FI-1",
        faceSimilarityCheck=True,
        # reference_img_path intentionally omitted
    )
except ValueError as e:
    print(f"Caught expected error: {e}")

---
## 7. FI-1 — Manual Polling (`auto_polling=False`)

Return immediately after upload, then poll manually when ready. Useful for web servers, background workers, or batched jobs where you don't want to block the main thread.

### 7a. Upload without blocking

In [ ]:
# Step 1 — fire upload, return immediately
upload_meta = client.face_intelligence(
    path="../data_samples/face_live_video/real/sample_real.mp4",
    model_type="FI-1",
    livenessCheck=True,
    auto_polling=False,        # do not block
)

mid = upload_meta["mid"]
print(f"Upload started.")
print(f"Media ID : {mid}")
print(f"Status   : {upload_meta['status']}  (processing started)")

In [ ]:
# Step 2 — do other work here, then poll when ready
print("Doing other work...")

# Poll manually
media = client.wait_for_media(mid, interval=5.0, timeout=600.0)
print(f"\nFinal Status : {media['status']}")
print(f"Liveness     : {media.get('isLiveness')}")

### 7b. Submit multiple jobs, then collect results

In [ ]:
import time

video_paths = [
    "../data_samples/face_live_video/real/sample_real.mp4",
    "../data_samples/face_live_video/fake/sample_fake.mp4",
]

# Submit all uploads without blocking
jobs = []
for path in video_paths:
    if not os.path.exists(path):
        continue
    meta = client.face_intelligence(
        path=path,
        model_type="FI-1",
        livenessCheck=True,
        auto_polling=False,
    )
    jobs.append({"path": path, "mid": meta["mid"]})
    print(f"Submitted: {os.path.basename(path)}  mid={meta['mid']}")

print(f"\nAll {len(jobs)} jobs submitted. Collecting results...")

# Collect results
for job in jobs:
    result = client.wait_for_media(job["mid"])
    print(f"  {os.path.basename(job['path']):30s}  isLiveness={result.get('isLiveness')}")

---
## 8. Media Management

Manage your media records on the Authenta platform.

### 8a. Get a specific media record

In [ ]:
# Replace with a real media ID from a previous upload
MEDIA_ID = "YOUR_MEDIA_ID"

media = client.get_media(MEDIA_ID)
print(f"Media ID   : {media['mid']}")
print(f"Status     : {media['status']}")
print(f"Model Type : {media.get('modelType')}")
print(f"Created At : {media.get('createdAt')}")

### 8b. List all media records

In [ ]:
# List all media (first page)
all_media = client.list_media()

items = all_media.get("items", [])
print(f"Total records returned: {len(items)}")
for item in items[:10]:   # show first 10
    print(f"  {item['mid']}  status={item['status']}  model={item.get('modelType')}")

In [ ]:
# Paginated listing
page_2 = client.list_media(page=2, pageSize=20)
for item in page_2.get("items", []):
    print(f"  {item['mid']}  {item['status']}")

### 8c. Delete a media record

In [ ]:
# Upload a file just to demonstrate deletion
upload_meta = client.upload_file(
    "../data_samples/face_similiar/person_1/A.jpeg",
    model_type="FI-1",
)
mid_to_delete = upload_meta["mid"]
print(f"Uploaded: {mid_to_delete}")

# Delete it
client.delete_media(mid_to_delete)
print(f"Deleted: {mid_to_delete}")

---
## 9. Visualization

Generate visual overlays — heatmaps and bounding boxes — to interpret detection results.

### 9a. Heatmap for AC-1 (Image)

In [ ]:
from authenta.visualization import save_heatmap

media = client.process("../data_samples/nano_img2.png", model_type="AC-1")

os.makedirs("results", exist_ok=True)
save_heatmap(
    media=media,
    out_path="results/ac1_heatmap.jpg",
    model_type="AC-1",
)
print("Heatmap saved to results/ac1_heatmap.jpg")

In [ ]:
# Display the heatmap inline
from IPython.display import Image as IPImage
IPImage("results/ac1_heatmap.jpg", width=600)

### 9b. Heatmap for DF-1 (Video — per participant)

In [ ]:
media = client.process("../data_samples/test_00000121.mp4", model_type="DF-1")

os.makedirs("results", exist_ok=True)

# Saves one heatmap video per participant:
#   results/heatmap_p0.mp4, results/heatmap_p1.mp4, ...
save_heatmap(
    media=media,
    out_path="./results",
    model_type="DF-1",
)
print("Heatmap videos saved to ./results/")

### 9c. Bounding Box Video (DF-1)

In [ ]:
from authenta.visualization import save_bounding_box_video

SRC_VIDEO = "../data_samples/test_00000121.mp4"
media = client.process(SRC_VIDEO, model_type="DF-1")

save_bounding_box_video(
    media,
    src_video_path=SRC_VIDEO,
    out_video_path="results/df1_annotated.mp4",
)
print("Annotated video saved to results/df1_annotated.mp4")

---
## 10. Async Client Examples

All services are available in the async client via `AsyncAuthentaClient`. Use `async with` to manage the HTTP session.

### 10a. AC-1 async detection

In [ ]:
import asyncio
from authenta.async_authenta_client import AsyncAuthentaClient

async def ac1_async():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as client:
        media = await client.process("../data_samples/nano_img2.png", model_type="AC-1")
        print(f"Status  : {media['status']}")
        print(f"Is Fake : {media.get('fake')}")

await ac1_async()

### 10b. DF-1 async detection

In [ ]:
async def df1_async():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as client:
        media = await client.process("../data_samples/test_00000121.mp4", model_type="DF-1")
        print(f"Status  : {media['status']}")
        print(f"Is Fake : {media.get('fake')}")

await df1_async()

### 10c. FI-1 liveness check (async)

In [ ]:
async def fi1_liveness_async():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as client:
        media = await client.process_FI(
            path="../data_samples/face_live_video/real/sample_real.mp4",
            model_type="FI-1",
            livenessCheck=True,
        )
        print(f"Status   : {media['status']}")
        print(f"Liveness : {media.get('isLiveness')}")

await fi1_liveness_async()

### 10d. FI-1 face swap detection (async)

In [ ]:
async def fi1_faceswap_async():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as client:
        media = await client.process_FI(
            path="../data_samples/faceswap/fake/sample_swap.mp4",
            model_type="FI-1",
            faceSwapCheck=True,
        )
        print(f"Status    : {media['status']}")
        print(f"Face Swap : {media.get('isDeepFake')}")

await fi1_faceswap_async()

### 10e. FI-1 face similarity (async)

In [ ]:
async def fi1_similarity_async():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as client:
        media = await client.process_FI(
            path="../data_samples/face_similiar/person_1/A.jpeg",
            model_type="FI-1",
            faceSimilarityCheck=True,
        )
        print(f"Status  : {media['status']}")
        print(f"Similar : {media.get('isSimilar')}")
        print(f"Score   : {media.get('similarityScore')}")

await fi1_similarity_async()

### 10f. FI-1 manual polling (async, `auto_polling=False`)

In [ ]:
async def fi1_manual_poll_async():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as client:
        # Step 1 — upload without blocking
        upload_meta = await client.process_FI(
            path="../data_samples/face_live_video/real/sample_real.mp4",
            model_type="FI-1",
            livenessCheck=True,
            auto_polling=False,
        )
        mid = upload_meta["mid"]
        print(f"Uploaded. Media ID: {mid}")

        # Step 2 — poll when ready
        media = await client.wait_for_media(mid)
        print(f"Status   : {media['status']}")
        print(f"Liveness : {media.get('isLiveness')}")

await fi1_manual_poll_async()

### 10g. Batch async processing (parallel)

In [ ]:
async def batch_async():
    video_paths = [
        "../data_samples/face_live_video/real/sample_real.mp4",
        "../data_samples/face_live_video/fake/sample_fake.mp4",
    ]
    existing = [p for p in video_paths if os.path.exists(p)]

    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as client:
        tasks = [
            client.process_FI(path=p, model_type="FI-1", livenessCheck=True)
            for p in existing
        ]
        results = await asyncio.gather(*tasks, return_exceptions=True)

        for path, result in zip(existing, results):
            name = os.path.basename(path)
            if isinstance(result, Exception):
                print(f"  [FAILED] {name}: {result}")
            else:
                print(f"  [OK]     {name:35s}  isLiveness={result.get('isLiveness')}")

await batch_async()

### 10h. Async media management

In [ ]:
async def manage_media_async():
    async with AsyncAuthentaClient(
        base_url=BASE_URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
    ) as client:
        # List all media
        all_media = await client.list_media(page=1, pageSize=10)
        print(f"Records: {len(all_media.get('items', []))}")
        for item in all_media.get("items", [])[:5]:
            print(f"  {item['mid']}  {item['status']}")

        # Get a specific record
        # media = await client.get_media("YOUR_MEDIA_ID")

        # Delete a specific record
        # await client.delete_media("YOUR_MEDIA_ID")

await manage_media_async()